# Model_ENSOclim_AkaikeCoeff-conservative

July 2025

Code by Caroline Juang (c.juang@columbia.edu)

This model will use **ENSO to predict climate variables**


In [1]:
# import
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import joblib # for saving models

## User input

In [2]:
# customize seasons for climate variables

# customize number of rolling periods
ant_years = 1 # antecedent years to include (2 antecedent years + current year)
ant_season = 3 # n+1 of months to include in each period (e.g. input 2 would mean 3 months)

firstyear = 1984 # first year of data
finalyear = 2022 # final year of data (should be same as burned area)
time_length = int(finalyear-firstyear+1) # get length of timeseries

# SST gradient: 
# this is the OBSERVED SST data to TRAIN the model on
# Read the climate index from the .txt file
with open('0_climindname.txt', 'r') as f:
    climindname = f.read().strip()
print(climindname +' will be used for the SST gradient')

# importing data string
directory = 'your_directory'
#data_string = 'data\\'
data_string = 'data//'
#model_string = 'model\\'+climindname+'\\'
model_string = 'model//'+climindname+'//'

patch125-155_nino3-34 will be used for the SST gradient


In [3]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

## Coefficients to Exclude

From `Model_ENSOClim_CoeffCheck`, and `Model_ENSOClim_CoeffCheck-avg`

Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

This is a correlation checker, which tells us which variables to exclude from the ENSO-climate model because the change in SST results in a change in the correlation coefficient between the variables.

In [4]:
# Model-CoeffCheck
# Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

with open(model_string + "coeffExclude_sst_all_ecoprovinces.txt", "r") as f:
    coeffExall = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sst_for_ecoprovinces.txt", "r") as f:
    coeffExfor = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sst_non_ecoprovinces.txt", "r") as f:
    coeffExnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
icoeffExall = [i for i, e in 
               enumerate(coeffExall) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffExfor = [i for i, e in 
               enumerate(coeffExfor) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffExnon = [i for i, e in 
               enumerate(coeffExnon) if "+++" in e]
# add in last index
icoeffExall.append(len(coeffExall)+1)
icoeffExfor.append(len(coeffExfor)+1)
icoeffExnon.append(len(coeffExnon)+1)

In [5]:
# Model_CoeffCheck-avg
# (average: average of prior- and concurrent-season gSST)
# Get the start and end of ecoregions from the txt files. txt files of variables to exclude from predicting climate.

with open(model_string + "coeffExclude_sstavg_all_ecoprovinces.txt", "r") as f:
    coeffavgExall = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sstavg_for_ecoprovinces.txt", "r") as f:
    coeffavgExfor = [line.strip() for line in f]
f.close()
with open(model_string + "coeffExclude_sstavg_non_ecoprovinces.txt", "r") as f:
    coeffavgExnon = [line.strip() for line in f]
f.close()

# get indices of where each ecoregion's outputs begin
icoeffavgExall = [i for i, e in 
               enumerate(coeffavgExall) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffavgExfor = [i for i, e in 
               enumerate(coeffavgExfor) if "+++" in e]
# get ecoregions so iteration is not manual
icoeffavgExnon = [i for i, e in 
               enumerate(coeffavgExnon) if "+++" in e]
# add in last index
icoeffavgExall.append(len(coeffavgExall)+1)
icoeffavgExfor.append(len(coeffavgExfor)+1)
icoeffavgExnon.append(len(coeffavgExnon)+1)

In [6]:
# get the SST inputs needed
def modelinputsst_string(inputlist, varname):
    """
    Requirements: 
    inputlist = list of the Model_ENSOclim_Akaike model results (locally defined)
    ithisecoreg = the ecoregion name (globally defined)
    ithisecoregend = the next ecoregion name 
        in the list (globally defined)
    varname = the climate variable name (locally defined)
    """
    # narrow inputs list to ecoregion
    modellist = inputlist[ithisecoreg:ithisecoregend+1]
    # get all PREDICTING (where inputs start)
    iinputs = [i for i, e in 
               enumerate(modellist) if "PREDICTING" in e]
    # extract model inputs
    istart = modellist.index('PREDICTING ' + varname)
    if len(iinputs)>((iinputs.index(istart))+1): # 
        iend = iinputs[(iinputs.index(istart))+1]
        modelinputs = modellist[istart+1:iend] # range in list
    else:
        modelinputs = modellist[istart+1:] # last predictor to end of list
    return modelinputs

In [7]:
# calculate AICc for variables

def getAICc(rvalue, nyears, npredictors):
    """
    Calculate the AICc, which is the Akaike Information Criterion (AIC)
    with bias correction for small sample sizes (AICc). 
    Inputs are the variables needed for the calculation. 
    Output is a single value of the AICc.
    r = r-value calculated from a Pearson Correlation
    nyears = number of years in the timeseries
    npredictors = number of x-variables used in the model
    """
    # calculate AIC
    aic = nyears * np.log(1-rvalue**2) + 2*(npredictors+1)
    # calculate AIC with bias correction for small sample size
    aicc = aic + (2*(npredictors+1)*(npredictors+2)) / (nyears - npredictors - 2)
    return aicc

## import ecoregion data

In [8]:
# import western US study area
westUS_string = '12km\\study_area\\westUS.nc'
westUS_string = '12km//study_area//westUS.nc'
westUS = xr.open_dataset(directory + westUS_string, engine='netcdf4')
westUS = westUS.westUS

# import forest
forest_string = '12km\\landcover\\US_ForestType_Ruefenacht\\forest_type_frac.nc'
forest_string = '12km//landcover//US_ForestType_Ruefenacht//forest_type_frac.nc'
forest = xr.open_dataset(directory + forest_string, engine='netcdf4')
forest = forest.forest_type_frac.sum(dim='ftype') # sum over all forest types

# create forest and nonforest area 
forest = westUS*forest
nonforest = westUS*(1-forest)
# add province as dimension to the westUS, to combine with ecoregions
westUS = westUS.expand_dims({'province':[0]})

In [9]:
# Bailey's ecoprovinces
ecoreg_string = '12km\\landcover\\Ecoregions_EPA\\bailey_ecoprovince.nc'
ecoreg_string = '12km//landcover//Ecoregions_EPA//bailey_ecoprovince.nc'
ecoreg = xr.open_dataset(directory + ecoreg_string, engine='netcdf4')
print(ecoreg.bailey_ecoprovince.province.legend)

# add in entire western US
ecoreg = xr.concat([westUS, ecoreg.bailey_ecoprovince], dim='province')
# remove the ecoregion that does not overlap with the western US
ecoreg = ecoreg.sel(province=province_num)

1: American Semi-Desert and Desert Province, 2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province, 3: Black Hills Coniferous Forest Province, 4: California Coastal Chapparral Forest and Shrub Province, 5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province, 6: California Coastal Steppe-Mixed Forest-Redwood Forest Province, 7: California Dry Steppe Province, 8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province, 9: Chihuahuan Semi-Desert Province, 10: Colorado Plateau Semi-Desert Province, 11: Great Plains Steppe Province, 12: Great Plains-Palouse Dry Steppe Province, 13: Intermountain Semi-Desert Province, 14: Intermountain Semi-Desert and Desert Province, 15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province, 16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province, 17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province, 18: Pacific 

In [10]:
# create strings of the types of forests, remove #5
province_names = ['1: American Semi-Desert and Desert Province',
                  '2: Arizona-New Mexico Mountains Semi-Desert-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '3: Black Hills Coniferous Forest Province',
                  '4: California Coastal Chapparral Forest and Shrub Province',
                  '5: California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow Province',
                  '6: California Coastal Steppe-Mixed Forest-Redwood Forest Province',
                  '7: California Dry Steppe Province',
                  '8: Cascade Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '9: Chihuahuan Semi-Desert Province',
                  '10: Colorado Plateau Semi-Desert Province',
                  '12: Great Plains-Palouse Dry Steppe Province',
                  '13: Intermountain Semi-Desert Province',
                  '14: Intermountain Semi-Desert and Desert Province',
                  '15: Middle Rocky Mountain Steppe-Coniferous Forest-Alpine Meadow Province',
                  '16: Nevada-Utah Mountains-Semi-Desert-Coniferous Forest-Alpine Meadow Province',
                  '17: Northern Rocky Mountain Forest-Steppe-Coniferous Forest-Alpine Meadow Province',
                  '18: Pacific Lowland Mixed Forest Province',
                  '19: Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow Province',
                  '20: Southern Rocky Mountain Steppe-Open Woodland-Coniferous Forest-Alpine Meadow Province',
                  '21: Southwest Plateau and Plains Dry Steppe and Shrub Province']
province_num = [item for item in range(len(province_names)+1+1)]
province_num.remove(11) # remove empty ecoregion - no overlap btwn westUS map and ecoregion
dfnames = ['allwestUS', 'ecoprov1', 'ecoprov2', 'ecoprov3', 'ecoprov4', 'ecoprov5', 
           'ecoprov6', 'ecoprov7', 'ecoprov8', 'ecoprov9', 'ecoprov10', 
           'ecoprov12','ecoprov13','ecoprov14','ecoprov15',
           'ecoprov16','ecoprov17','ecoprov18','ecoprov19','ecoprov20','ecoprov21']

# import SST gradient data
from `Data_CreateModelData`

In [11]:
# import SST gradient
climind_seasons = pd.read_csv(data_string + 'sstgrad_seasons_' +climindname+'_82_y.txt').set_index('Unnamed: 0')
climind_seasons.drop(labels=['y-1 mo 1-3', 'y-1 mo 4-6', 'y-1 mo 7-9'], axis=1, inplace=True)

# import average of prior and current-season gSST
climind_seasonsavg = pd.read_csv(data_string + 'sstgrad_seasons_' +climindname+'_82_y_avgcurr-prior.txt').set_index('Unnamed: 0')
print('import ' +climindname+'_82_y_avgcurr-prior')
climind_seasonsavg.drop(labels=['y-1 mo 1-6', 'y-1 mo 4-9'], axis=1, inplace=True)

# CHECK: n columns is the same between same-season and averaged climate indices
if len(climind_seasons.columns) == len(climind_seasonsavg.columns):
    print('number of seasons present is correct')
else:
    raise RuntimeError("Number of seasons columns is mismatched. Stopping the kernel.")

import patch125-155_nino3-34_82_y_avgcurr-prior
number of seasons present is correct


# Import climate data
from `Data_CreateModelData`

In [12]:
# import model data for climate

dfframesall = {}
dfframesfor = {}
dfframesnon = {}

for iecoreg in np.arange(len(province_num)):
    filename = data_string + 'climatefull_ecoprovinces_'
    dfframesall[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_all.txt').set_index('Unnamed: 0')
    dfframesfor[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_for.txt').set_index('Unnamed: 0')
    dfframesnon[dfnames[iecoreg]] = pd.read_csv(filename + dfnames[iecoreg]+'_non.txt').set_index('Unnamed: 0')

# Model

In [13]:
# storage for each model's variable indices
dfmodelall = {}
dfmodelfor = {}
dfmodelnon = {}

# Run model for ALL

In [14]:
# regress all potential predictors against climate, for ALL area

# for the coefficient to exclude
landtypeinput = coeffExall
landtypeiinput = icoeffExall
landtypeavginput = coeffavgExall
landtypeavgiinput = icoeffavgExall

print('ALL LANDCOVER')
land = 'all'
modeloutputfile = model_string + 'modeloutput_sst_all_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesall['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ ALL LANDCOVER model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)
        
        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesall[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr = pearsonr(tmpclim.flatten(), tmppredict.flatten())[0]

            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)
            
            # store AICC and r-value
            aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: # as long as CoeffExclude not used # new, 8/23/2024
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))

    dfmodelall[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list
f.close()

ALL LANDCOVER
++++++++ ALL LANDCOVER model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-12.405	0.523	FINAL MODEL v1	Coef =[[-0.546]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-11.708	0.575	FINAL MODEL v2	Coef =[[-0.685]]
PREDICTING rh y0 mo 7-9
y0 mo 4-9	0.803	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.29	FINAL MODEL v1	Coef =[[0]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	1.032	CASE 3 EXCL AVG
y0 mo 10-12	1.032	0.28	FINAL MODEL v1	Coef =[[-0.22]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-5.432	0.413	FINAL MODEL v1	Coef =[[0.434]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-2.181	CASE 2 EXCL CONCURR
y0 mo 1-6	-2.181	0.387	FINAL MODEL v2	Coef =[[0.462]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	0.758	CASE 3 EXCL AVG
y0 mo 7-9	0.758	0.274	FINAL MODEL v1	Coef =[[0.278]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	4.323	CASE 3 EXCL AVG
y0 mo 10-12	4.323	0.038	FINAL MODEL v1	Coef =[[-0.004]]
PRE

y0 mo 10-12	-3.522	0.419	FINAL MODEL v1	Coef =[[0.33]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	3.964	CASE 3 EXCL AVG
y0 mo 1-3	3.964	0.12	FINAL MODEL v1	Coef =[[-0.088]]
PREDICTING tmin y0 mo 4-6
y0 mo 1-6	-4.689	CASE 2 EXCL CONCURR
y0 mo 1-6	-4.689	0.449	FINAL MODEL v2	Coef =[[0.535]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	1.608	CASE 3 EXCL AVG
y0 mo 7-9	1.608	0.301	FINAL MODEL v1	Coef =[[0.244]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	0.437	CASE 3 EXCL AVG
y0 mo 10-12	0.437	0.271	FINAL MODEL v1	Coef =[[0.238]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	2.31	CASE 3 EXCL AVG
y0 mo 1-3	2.31	0.17	FINAL MODEL v1	Coef =[[0.207]]
PREDICTING tmean y0 mo 4-6
y0 mo 1-6	-7.043	CASE 2 EXCL CONCURR
y0 mo 1-6	-7.043	0.497	FINAL MODEL v2	Coef =[[0.593]]
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	1.55	CASE 3 EXCL AVG
y0 mo 7-9	1.55	0.289	FINAL MODEL v1	Coef =[[0.246]]
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	-2.659	0.385	FINAL MODEL v1	Coef =[[0.313]]
PREDICTING vpd y0 mo 1-3
y0 mo 1-3	-8.705	0.482	FINAL MODEL v1	Coef

y0 mo 1-3	-16.742	0.597	FINAL MODEL v1	Coef =[[-0.597]]
PREDICTING prec y0 mo 4-6
y0 mo 1-6	1.526	CASE 2 EXCL CONCURR
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.26	FINAL MODEL v1	Coef =[[0]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	4.323	CASE 3 EXCL AVG
y0 mo 7-9	4.323	0.011	FINAL MODEL v1	Coef =[[0.005]]
PREDICTING prec y0 mo 10-12
y0 mo 10-12	2.17	CASE 3 EXCL AVG
y0 mo 10-12	2.17	0.262	FINAL MODEL v1	Coef =[[-0.179]]
++++++++ ALL LANDCOVER model for ecoprov2	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-17.739	0.614	FINAL MODEL v1	Coef =[[-0.608]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-11.888	0.577	FINAL MODEL v2	Coef =[[-0.688]]
PREDICTING rh y0 mo 7-9
y0 mo 7-9	4.315	CASE 3 EXCL AVG
y0 mo 7-9	4.315	0.01	FINAL MODEL v1	Coef =[[-0.014]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	-6.221	0.462	FINAL MODEL v1	Coef =[[-0.377]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-3.836	0.393	FINAL MODEL v1	Coef =[[0.401]]
PRED

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 4-6	2.155	0.101	FINAL MODEL v1	Coef =[[0.3]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-1.506	CASE 2 EXCL CONCURR
y0 mo 4-9	-1.506	0.368	FINAL MODEL v2	Coef =[[0.439]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	2.709	CASE 3 EXCL AVG
y0 mo 10-12	2.709	0.157	FINAL MODEL v1	Coef =[[0.156]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	4.182	CASE 3 EXCL AVG
y0 mo 1-3	4.182	0.075	FINAL MODEL v1	Coef =[[-0.056]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	4.049	CASE 3 EXCL AVG
y0 mo 4-6	4.049	0.07	FINAL MODEL v1	Coef =[[-0.108]]
PREDICTING wetdays y0 mo 7-9
y0 mo 4-9	0.379	CASE 2 EXCL CONCURR
y0 mo 4-9	0.379	0.306	FINAL MODEL v2	Coef =[[-0.365]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	3.556	CASE 3 EXCL AVG
y0 mo 10-12	3.556	0.159	FINAL MODEL v1	Coef =[[0.108]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	-5.565	0.416	FINAL MODEL v1	Coef =[[0.437]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-5.304	0.463	FINAL MODEL v2	Coef =[[0.551]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	4.318	CASE 3 EXCL AVG
y0 mo 7-9	4.318	0.007	FINAL MOD

...

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 7-9	3.971	0.203	FINAL MODEL v1	Coef =[[-0.089]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	4.286	CASE 3 EXCL AVG
y0 mo 10-12	4.286	0.088	FINAL MODEL v1	Coef =[[-0.024]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-1.387	0.316	FINAL MODEL v1	Coef =[[0.341]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-1.794	CASE 2 EXCL CONCURR
y0 mo 1-6	-1.794	0.377	FINAL MODEL v2	Coef =[[0.449]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	2.639	CASE 3 EXCL AVG
y0 mo 7-9	2.639	0.189	FINAL MODEL v1	Coef =[[0.193]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	4.012	CASE 3 EXCL AVG
y0 mo 10-12	4.012	0.009	FINAL MODEL v1	Coef =[[-0.069]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	4.07	CASE 3 EXCL AVG
y0 mo 1-3	4.07	0.004	FINAL MODEL v1	Coef =[[0.074]]
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	4.022	CASE 3 EXCL AVG
y0 mo 4-6	4.022	0.213	FINAL MODEL v1	Coef =[[0.113]]
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	2.416	CASE 3 EXCL AVG
y0 mo 7-9	2.416	0.273	FINAL MODEL v1	Coef =[[0.205]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	4.075	CASE 3 EXCL AVG
y0 mo 1

y0 mo 1-6	-4.975	0.455	FINAL MODEL v2	Coef =[[0.543]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-2.946	0.408	FINAL MODEL v2	Coef =[[0.486]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-1.095	0.334	FINAL MODEL v1	Coef =[[0.278]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	4.247	CASE 3 EXCL AVG
y0 mo 1-3	4.247	0.007	FINAL MODEL v1	Coef =[[-0.041]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	-1.333	0.344	FINAL MODEL v1	Coef =[[-0.474]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	3.243	CASE 3 EXCL AVG
y0 mo 7-9	3.243	0.152	FINAL MODEL v1	Coef =[[-0.155]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	4.313	CASE 3 EXCL AVG
y0 mo 10-12	4.313	0.043	FINAL MODEL v1	Coef =[[-0.013]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	-6.264	0.471	FINAL MODEL v1	Coef =[[0.45]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-6.829	0.493	FINAL MODEL v2	Coef =[[0.588]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	4.191	CASE 3 EXCL AVG
y0 mo 7-9	4.191	0.119	FINAL MODEL v1	Coef =[[-0.055]]
PREDICTING wind y0 mo 10-12
y0 mo 10-12	-5.81	0.461	FINAL MODEL v1	Coef =[[0.3

y0 mo 7-9	2.956	0.139	FINAL MODEL v1	Coef =[[0.175]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	-0.871	0.333	FINAL MODEL v1	Coef =[[0.273]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	-4.779	0.425	FINAL MODEL v1	Coef =[[0.421]]
PREDICTING tmax y0 mo 4-6
y0 mo 1-6	-7.092	0.498	FINAL MODEL v2	Coef =[[0.594]]
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	3.585	CASE 3 EXCL AVG
y0 mo 7-9	3.585	0.076	FINAL MODEL v1	Coef =[[0.129]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	-8.284	0.51	FINAL MODEL v1	Coef =[[0.407]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	3.708	CASE 3 EXCL AVG
y0 mo 1-3	3.708	0.133	FINAL MODEL v1	Coef =[[-0.115]]
PREDICTING tmin y0 mo 4-6
y0 mo 1-6	-11.457	0.571	FINAL MODEL v2	Coef =[[0.681]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	3.102	CASE 3 EXCL AVG
y0 mo 7-9	3.102	0.158	FINAL MODEL v1	Coef =[[0.165]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	1.82	CASE 3 EXCL AVG
y0 mo 10-12	1.82	0.253	FINAL MODEL v1	Coef =[[0.193]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	1.668	CASE 3 EXCL AVG
y0 mo 1-3	1.668	0.231	FINAL MO

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/2871967539.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

# Run Model for FOREST

In [15]:
# regress all potential predictors against climate, for FOREST area
print('FOREST')
land = 'for'

# for the coefficient to exclude
landtypeinput = coeffExfor
landtypeiinput = icoeffExfor
landtypeavginput = coeffavgExfor
landtypeavgiinput = icoeffavgExfor

modeloutputfile = model_string + 'modeloutput_sst_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesfor['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ FOREST model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)

        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesfor[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr, tmpp = pearsonr(tmpclim.flatten(), tmppredict.flatten())
            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)
            if tmpp>=0.1:
                aicc[i] = 99 # set to exclude if p-value is not significant
                # conservative version (2/5/2025)
            else:
                # store AICC and r-value
                aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: 
            # as long as CoeffExclude not used AND (conservative) p-value<0.1
            # new, 8/23/2024 + 2/5/2025
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))

    dfmodelfor[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list
f.close()

FOREST
++++++++ FOREST model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-11.129	0.493	FINAL MODEL v1	Coef =[[-0.529]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-7.806	0.511	FINAL MODEL v2	Coef =[[-0.61]]
PREDICTING rh y0 mo 7-9
y0 mo 4-9	0.937	CASE 2 EXCL CONCURR
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.285	FINAL MODEL v1	Coef =[[0]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.207	FINAL MODEL v1	Coef =[[0]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	0.131	CASE 3 EXCL AVG
y0 mo 1-3	0.131	0.267	FINAL MODEL v1	Coef =[[0.295]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	0.983	CASE 2 EXCL CONCURR
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.283	FINAL MODEL v1	Coef =[[0]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	1.295	CASE 3 EXCL AVG
y0 mo 7-9	1.295	0.271	FINAL MODEL v1	Coef =[[0.257]]
PREDICT

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 1-3	99.0	0.166	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.149	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	1.386	CASE 3 EXCL AVG
y0 mo 7-9	1.386	0.322	FINAL MODEL v1	Coef =[[0.253]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.202	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 1-3
y-1 mo 10-3	-2.068	0.384	FINAL MODEL v2	Coef =[[-0.324]]
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.005	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.247	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.15	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmean y0 mo

y0 mo 7-9	99.0	0.156	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 10-12
y0 mo 7-12	-3.38	0.419	FINAL MODEL v2	Coef =[[-0.367]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.239	FINAL MODEL v1	Coef =[[0]]
PREDICTING wind y0 mo 4-6
y0 mo 4-6	1.326	CASE 3 EXCL AVG
y0 mo 4-6	1.326	0.27	FINAL MODEL v1	Coef =[[0.351]]
PREDICTING wind y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.093	FINAL MODEL v1	Coef =[[0]]
PREDICTING wind y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.016	FINAL MODEL v1	Coef =[[0]]
PREDICTING prec y0 mo 1-3
y0 mo 1-3	-17.648	0.604	FINAL MODEL v1	Coef =[[-0.607]]
PREDICTING prec y0 mo 4-6
y0 mo 1-6	0.276	CASE 2 EXCL CONCURR
y0 mo 1-6	0.276	0.31	FINAL MODEL v2	Coef =[[-0.37]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.052	F

y0 mo 7-9	99.0	0.102	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	-6.441	0.467	FINAL MODEL v1	Coef =[[0.38]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.015	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 4-6
y0 mo 1-6	-10.609	0.558	FINAL MODEL v2	Coef =[[0.665]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	1.256	CASE 3 EXCL AVG
y0 mo 7-9	1.256	0.276	FINAL MODEL v1	Coef =[[0.259]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	-1.347	0.35	FINAL MODEL v1	Coef =[[0.284]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	-0.138	CASE 3 EXCL AVG
y0 mo 1-3	-0.138	0.286	FINAL MODEL v1	Coef =[[0.303]]
PREDICTING tmean y0 mo 4-6
y0 mo 1-6	-14.833	0.617	FINAL MODEL v2	Coef =[[0.735]]
PREDICTING tmean y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.193	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmean y0 mo 10-12
y0 mo 10-12	-6.801	0.474	FINAL MODEL v1	Coef =[[0.385]]
PREDICTING vpd y0 mo 1

...

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 7-9	99.0	0.111	FINAL MODEL v1	Coef =[[0]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.059	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.055	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 4-6
y0 mo 4-6	0.987	CASE 3 EXCL AVG
y0 mo 4-6	0.987	0.306	FINAL MODEL v1	Coef =[[0.369]]
PREDICTING tmax y0 mo 7-9
y0 mo 4-9	-2.106	0.385	FINAL MODEL v2	Coef =[[0.459]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	-1.997	0.367	FINAL MODEL v1	Coef =[[0.299]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.174	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 4-6
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.178	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	1.126	CASE 3 EXCL AVG
y0 mo 7-9	1.126	0.294	FINAL MODE

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

y0 mo 1-3	-0.186	0.274	FINAL MODEL v1	Coef =[[0.305]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-5.549	0.468	FINAL MODEL v2	Coef =[[0.557]]
PREDICTING vpd y0 mo 7-9
y0 mo 4-9	-1.834	CASE 2 EXCL CONCURR
y0 mo 4-9	-1.834	0.378	FINAL MODEL v2	Coef =[[0.45]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-1.1	0.335	FINAL MODEL v1	Coef =[[0.279]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.01	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	-0.928	0.335	FINAL MODEL v1	Coef =[[-0.458]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.115	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.048	FINAL MODEL v1	Coef =[[0]]
PREDICTING wind y0 mo 1-3
y0 mo 1-3	-6.137	0.472	FINAL MODEL v1	Coef =[[0.448]]
PREDICTING wind y0 mo 4-6
y0 mo 1-6	-6.001	0.477	FINAL MOD

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/1934660591.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = 

# Run Model for NONFOREST

In [16]:
# regress all potential predictors against climate, for NONFOREST area

print('NONFOREST')
land = 'non'

# for the coefficient to exclude
landtypeinput = coeffExnon
landtypeiinput = icoeffExnon
landtypeavginput = coeffavgExnon
landtypeavgiinput = icoeffavgExnon

modeloutputfile = model_string + 'modeloutput_sst_'+land+'.txt'
f = open(modeloutputfile, 'w') # print to this file
labels = dfframesnon['allwestUS'].columns.values # each climate variable
for iregion in np.arange(len(province_num)): # iterate through ecoregion

    print('++++++++ NONFOREST model for ' + dfnames[iregion]+ '\t AICC \t r-value \tCoeff Case \tAICc Case \tCoefficient++++++')
    f.write('+++'+dfnames[iregion]+'\n')
    for i, label in enumerate(labels): # iterate through climate variables
        tmpaicc = [] # empty list of aicc values for each regression fit
        tmpi = [] # empty list of indices for variables

        # retrieve coefficients to exclude
        ithisecoreg = landtypeinput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeiinput[landtypeiinput.index(ithisecoreg)+1]-1
        modelexclude = modelinputsst_string(landtypeinput, label)

        ithisecoreg = landtypeavginput.index('+++'+dfnames[iregion])
        ithisecoregend = landtypeavgiinput[landtypeavgiinput.index(ithisecoreg)+1]-1
        modelexcludeavg = modelinputsst_string(landtypeavginput, label)

        # VARIABLES TO COMPARE
        # each climate variable
        tmpclim = np.asarray(dfframesnon[dfnames[iregion]][label]).reshape(-1,1)
        # associated concurrent-season climate index (ENSO or other)
        strseason = label.split(' ',1)[1] # get season label
        matchseason = climind_seasons.columns.isin([strseason]) # True/False array
        tmpclimind = np.asarray(climind_seasons.loc[:,matchseason])
        imatchseason = np.where(matchseason)[0]
        tmpclimindavg = np.asarray(climind_seasonsavg.iloc[:,imatchseason])
        # grab average of (previous season+current season)         
        iprevseason = int(imatchseason)-1
        tmpclimindprev = np.asarray(climind_seasons.iloc[:,iprevseason]).reshape(-1,1)
        strprevseason = climind_seasons.columns[iprevseason] # label

        print('PREDICTING '+label)
        f.write('PREDICTING '+label)
        f.write('\n')

        # STEP 1 linear regression, compare concurrent, or prior gSST
        sstlist = [tmpclimind, tmpclimindavg]
        aicc = np.zeros(len(sstlist))
        
        for i,selectclimind in enumerate(sstlist):
            reg = LinearRegression() # least squares
            reg.fit(selectclimind, tmpclim)
            tmppredict = reg.predict(selectclimind)
            tmpr, tmpp = pearsonr(tmpclim.flatten(), tmppredict.flatten())
            
            # calculate AIC
            nyears = len(selectclimind)
            npredictors = reg.n_features_in_
            tmpgetaicc = getAICc(tmpr, nyears, npredictors)

            if tmpp>=0.1:
                aicc[i] = 99 # set to exclude if p-value is not significant
                # conservative version (2/5/2025)
            else:
                # store AICC and r-value
                aicc[i] = tmpgetaicc

        # check if (1) current-season correlation is in CoeffExclude
        if (len(modelexclude)==1) & (strseason in modelexclude):
            aicc[0] = 99 # set to massive positive if so
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(aicc[1], decimals=3))
                 + '\tCASE 2 EXCL CONCURR')
        # check if (2) avgcurrent+prior is in CoeffExclude
        if (len(modelexcludeavg)==1) & (strseason in modelexcludeavg):
            aicc[1] = 99
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(aicc[0], decimals=3))
                  + '\tCASE 3 EXCL AVG')
        
        # get minimum of the two values (iminaicc would be 0 or 1)
        [iminaicc, minaicc] = np.argmin(aicc), np.min(aicc)
        tmpi.append(iminaicc) # add min to list of variables
        tmpaicc.append(minaicc) # add aicc value to list
        
        # regress model again
        reg = LinearRegression() # least squares
        reg.fit(sstlist[iminaicc], tmpclim)
            
        if minaicc>98: # as long as CoeffExclude not used AND (conservative) p<0.1
            # new, 8/23/2024 + 2/5/2025
            # force change the coefficient and intercept
            reg.coef_ = np.asarray([[0]])
            reg.intercept_ = tmpclim.mean()
            print('\tAICC POSITIVE--CHANGE COEF' 
                  + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
            # store the modified model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        else: # if there is a negative min AICC, that is the new model
            # store the newest model
            filename = model_string + 'model_'+land+'_'+label+'_'+dfnames[iregion]+'.sav' # rewrite model and save
            joblib.dump(reg, filename)
        
        # record the season (in column name) to the file
        if (iminaicc==0):
            f.write(climind_seasons.columns[matchseason].tolist()[0])
            f.write('\n')
            print(climind_seasons.columns[matchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v1'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
        elif (iminaicc==1):
            f.write(climind_seasonsavg.columns[imatchseason].tolist()[0])
            f.write('\n')
            print(climind_seasonsavg.columns[imatchseason].tolist()[0] + '\t' + str(np.round(minaicc, decimals=3)) + 
              '\t' + str(np.round(tmpr, decimals=3)) + '\tFINAL MODEL v2'
             + '\tCoef ='+str(np.round(reg.coef_, decimals=3)))
    
    dfmodelnon[dfnames[iregion]] = [labels[i] for i in tmpi] # add this ecoregion's values to list

NONFOREST


++++++++ NONFOREST model for allwestUS	 AICC 	 r-value 	Coeff Case 	AICc Case 	Coefficient++++++
PREDICTING rh y0 mo 1-3
y0 mo 1-3	-11.945	0.522	FINAL MODEL v1	Coef =[[-0.54]]
PREDICTING rh y0 mo 4-6
y0 mo 1-6	-12.745	0.589	FINAL MODEL v2	Coef =[[-0.703]]
PREDICTING rh y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.289	FINAL MODEL v1	Coef =[[0]]
PREDICTING rh y0 mo 10-12
y0 mo 10-12	0.497	CASE 3 EXCL AVG
y0 mo 10-12	0.497	0.301	FINAL MODEL v1	Coef =[[-0.236]]
PREDICTING solar y0 mo 1-3
y0 mo 1-3	-7.845	0.461	FINAL MODEL v1	Coef =[[0.478]]
PREDICTING solar y0 mo 4-6
y0 mo 1-6	-3.097	0.411	FINAL MODEL v2	Coef =[[0.491]]
PREDICTING solar y0 mo 7-9
y0 mo 7-9	0.757	CASE 3 EXCL AVG
y0 mo 7-9	0.757	0.266	FINAL MODEL v1	Coef =[[0.278]]
PREDICTING solar y0 mo 10-12
y0 mo 10-12	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 10-12	99.0	0.103	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	A

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int

y0 mo 1-3	-2.578	0.337	FINAL MODEL v1	Coef =[[0.372]]
PREDICTING tmax y0 mo 4-6
y0 mo 1-6	-8.701	0.527	FINAL MODEL v2	Coef =[[0.628]]
PREDICTING tmax y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.243	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmax y0 mo 10-12
y0 mo 10-12	-3.489	0.418	FINAL MODEL v1	Coef =[[0.33]]
PREDICTING tmin y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0.122	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 4-6
y0 mo 1-6	-4.655	CASE 2 EXCL CONCURR
y0 mo 1-6	-4.655	0.448	FINAL MODEL v2	Coef =[[0.535]]
PREDICTING tmin y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.301	FINAL MODEL v1	Coef =[[0]]
PREDICTING tmin y0 mo 10-12
y0 mo 10-12	0.498	CASE 3 EXCL AVG
y0 mo 10-12	0.498	0.269	FINAL MODEL v1	Coef =[[0.236]]
PREDICTING tmean y0 mo 1-3
y0 mo 1-3	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 1-3	99.0	0

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int

...

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int

y0 mo 1-3	-12.481	0.567	FINAL MODEL v1	Coef =[[0.547]]
PREDICTING vpd y0 mo 4-6
y0 mo 1-6	-4.189	0.438	FINAL MODEL v2	Coef =[[0.522]]
PREDICTING vpd y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.086	FINAL MODEL v1	Coef =[[0]]
PREDICTING vpd y0 mo 10-12
y0 mo 10-12	-8.5	0.505	FINAL MODEL v1	Coef =[[0.41]]
PREDICTING wetdays y0 mo 1-3
y0 mo 1-3	-3.266	0.385	FINAL MODEL v1	Coef =[[-0.388]]
PREDICTING wetdays y0 mo 4-6
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 4-6	99.0	0.209	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.043	FINAL MODEL v1	Coef =[[0]]
PREDICTING wetdays y0 mo 10-12
y0 mo 10-12	-4.863	0.447	FINAL MODEL v1	Coef =[[-0.354]]
PREDICTING wind y0 mo 1-3
y-1 mo 10-3	-1.063	0.355	FINAL MODEL v2	Coef =[[0.299]]
PREDICTING wind y0 mo 4-6
y0 mo 4-6	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[

/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int

y0 mo 4-6	99.0	0.093	FINAL MODEL v1	Coef =[[0]]
PREDICTING prec y0 mo 7-9
y0 mo 7-9	99.0	CASE 3 EXCL AVG
	AICC POSITIVE--CHANGE COEF	Coef =[[0]]
y0 mo 7-9	99.0	0.097	FINAL MODEL v1	Coef =[[0]]
PREDICTING prec y0 mo 10-12
y0 mo 10-12	-2.655	0.388	FINAL MODEL v1	Coef =[[-0.313]]


/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1
/var/folders/bh/cplk6_2163d7cqx4vgqpkzz40000gn/T/ipykernel_56189/797654068.py:42: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  iprevseason = int(imatchseason)-1


In [17]:
# close files
f.close()
# print last time this model was updated
from datetime import datetime
now = datetime.now()
# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print('SST gradient used: '+climindname)
print("Model last run =", dt_string)

SST gradient used: patch125-155_nino3-34
Model last run = 26/06/2026 13:57:07
